In [3]:
# Install libraries
!pip install -q transformers datasets accelerate evaluate scikit-learn

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

# ==========================
# LOAD CSV
# ==========================
from google.colab import files

uploaded = files.upload()

df = pd.read_csv("IMDB Dataset.csv")   # Change filename if needed

print("Dataset Shape:", df.shape)
print(df.head())

# ==========================
# LABEL ENCODING
# ==========================

df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

df = df[["review", "label"]]

print("\nLabel Distribution:")
print(df["label"].value_counts())

# ==========================
# TRAIN TEST SPLIT
# ==========================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# ==========================
# TOKENIZER
# ==========================

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["review"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

# ==========================
# MODEL
# ==========================

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# ==========================
# METRIC
# ==========================

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

# ==========================
# TRAINING SETTINGS
# ==========================

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100
)

# ==========================
# TRAINER
# ==========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset.select(range(5000)),
    eval_dataset=test_dataset.select(range(1000)),
    compute_metrics=compute_metrics
)

# ==========================
# TRAIN
# ==========================

print("\nTraining Started...\n")

trainer.train()

# ==========================
# EVALUATE
# ==========================

print("\nEvaluating...\n")

results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)

# ==========================
# INFERENCE
# ==========================

print("\nInference Examples:\n")

samples = [
    "This movie was fantastic and inspiring.",
    "Worst movie ever.",
    "I enjoyed every minute of it.",
    "Waste of time."
]

for review in samples:

    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = outputs.logits.argmax().item()

    label = "Positive" if prediction == 1 else "Negative"

    print(f"{review} --> {label}")

Saving IMDB Dataset.csv to IMDB Dataset (2).csv
Dataset Shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Label Distribution:
label
1    25000
0    25000
Name: count, dtype: int64


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.375392,0.374670,0.840000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating...



Training Loss,Validation Loss,Epoch,Accuracy
0.375392,0.374670,1,0.840000



Evaluation Results:
{'eval_loss': 0.37466952204704285, 'eval_accuracy': 0.84}

Inference Examples:

This movie was fantastic and inspiring. --> Positive
Worst movie ever. --> Negative
I enjoyed every minute of it. --> Positive
Waste of time. --> Negative
